<a href="https://colab.research.google.com/github/whgusdn5221/comfycolab/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# [1] 시스템 기본 최적화
!apt -y update -qq
!wget https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4
!apt -y install -qq aria2

# [2] 필수 라이브러리 강제 주입 (매니저 & IP-Adapter 부품)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers triton mediapipe addict yapf fvcore omegaconf
!pip install -q gitpython insightface onnxruntime-gpu

# [3] ComfyUI 본체 및 필수 커스텀 노드 클린 설치
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# 매니저와 IP-Adapter 노드를 '신선한' 상태로 다시 받습니다.
!git clone https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/ComfyUI-Manager
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus

# [4] 구글 드라이브 마운트 및 모델 연결 (심볼릭 링크)
from google.colab import drive
drive.mount('/content/drive')

model_types = ["checkpoints", "clip_vision", "ipadapter", "vae", "loras", "upscale_models"]
for m_type in model_types:
    drive_path = f"/content/drive/MyDrive/ComfyUI/models/{m_type}"
    colab_path = f"/content/ComfyUI/models/{m_type}"
    if os.path.exists(drive_path):
        !rm -rf {colab_path}
        !ln -s {drive_path} {colab_path}
        print(f"✅ {m_type} 폴더 연결 완료")

# [5] 접속 주소 생성 (Cloudflare)
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64
import atexit, requests, subprocess, time, re
from random import randint
from threading import Timer
from queue import Queue

def cloudflared(port, metrics_port, output_queue):
    atexit.register(lambda p: p.terminate(), subprocess.Popen(['/content/cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
    attempts, tunnel_url = 0, None
    while attempts < 10 and not tunnel_url:
        time.sleep(3)
        try:
            # r"" 접두사를 붙여 SyntaxWarning을 완벽히 해결했습니다.
            tunnel_url = re.search(r"(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text).group("url")
        except:
            attempts += 1
    if not tunnel_url: raise Exception("Cloudflare 연결 실패")
    output_queue.put(tunnel_url)

output_queue, metrics_port = Queue(), randint(8100, 9000)
thread = Timer(2, cloudflared, args=(8188, metrics_port, output_queue))
thread.start()
thread.join()
print(f"\n🚀 접속 주소: {output_queue.get()}\n")

# [6] ComfyUI 가동
!python main.py --dont-print-server

In [ ]:
from google.colab import drive
drive.mount('/content/drive')